# Etapa 2 — Comparar as Capitais
## Estatística Descritiva por Função

## Contexto

Na etapa anterior calculamos a taxa de execução para todas as 26 capitais em todas as funções. Agora vamos comparar esses valores usando estatística descritiva para identificar padrões.

## Objetivo

Comparar as capitais usando **média, mediana, desvio padrão e coeficiente de variação** para identificar:

- Quem executa melhor em cada função
- Se a mediana confirma a média
- Qual função tem mais desigualdade entre capitais

## Perguntas que esta etapa responde

1. **"Quem executa melhor em Saúde? Em Educação?"**
2. **"A mediana confirma a média ou tem outlier distorcendo?"**
3. **"Qual função tem mais desigualdade entre as capitais?"**

## Fórmulas

**Média:**
$$Media(F) = \frac{\sum TaxaExecucao(C,F)}{n}$$

**Mediana:**
$$Mediana(F) = \frac{T_{13} + T_{14}}{2}$$

**Desvio Padrão Amostral:**
$$DP(F) = \sqrt{\frac{\sum (TaxaExecucao(C,F) - Media(F))^2}{n-1}}$$

**Coeficiente de Variação:**
$$CV(F) = \frac{DP(F)}{Media(F)} \times 100$$

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.banco.conexao_duckdb import conectar
from src.utils.constantes import CAMINHO_DUCKDB

In [3]:
# Conectar ao banco
con = conectar(CAMINHO_DUCKDB)
print(f'Conectado a: {CAMINHO_DUCKDB}')

Conectado a: c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\finbra.duckdb


## 1. Carregar Dados da Etapa 1

Vamos carregar os dados calculados na etapa anterior.

In [4]:
# Carregar dados da etapa 1
pasta_processados = Path.cwd().parent / 'data' / 'processed'

df_taxa_funcao = pd.read_parquet(pasta_processados / 'etapa1_taxa_por_funcao.parquet')
df_taxa_geral = pd.read_parquet(pasta_processados / 'etapa1_taxa_geral.parquet')

print(f'Taxa por funcao: {len(df_taxa_funcao)} registros')
print(f'Taxa geral: {len(df_taxa_geral)} capitais')
df_taxa_funcao.head()

Taxa por funcao: 541 registros
Taxa geral: 26 capitais


,Instituição,UF,Funcao,Pago,Empenhado,Taxa_Execucao
0,Prefeitura Municipal de Aracaju - SE,SE,01 - Legislativa,3.342494e+08,3.393389e+08,98.50
1,Prefeitura Municipal de Aracaju - SE,SE,04 - Administração,1.627963e+09,1.678003e+09,97.02
2,Prefeitura Municipal de Aracaju - SE,SE,06 - Segurança Pública,4.255507e+06,4.852577e+06,87.70
3,Prefeitura Municipal de Aracaju - SE,SE,08 - Assistência Social,3.315742e+08,3.585380e+08,92.48
4,Prefeitura Municipal de Aracaju - SE,SE,09 - Previdência Social,1.815940e+09,1.819511e+09,99.80


## 2. Ranking por Função

Para cada função, vamos ranquear as capitais pela taxa de execução.

In [9]:
# Funcao de ranking por funcao
def ranking_por_funcao(df, funcao):
    """Retorna ranking das capitais para uma funcao especifica."""
    df_funcao = df[df['Funcao'] == funcao].copy()
    df_funcao = df_funcao.dropna(subset=['Taxa_Execucao'])
    df_funcao = df_funcao.sort_values('Taxa_Execucao', ascending=False)
    df_funcao['Ranking'] = range(1, len(df_funcao) + 1)
    return df_funcao[['Ranking', 'Instituição', 'UF', 'Taxa_Execucao']]

# Exemplo: Saude
print('=== RANKING SAUDE ===')
ranking_saude = ranking_por_funcao(df_taxa_funcao, '10 - Saúde')
ranking_saude.head(10)

=== RANKING SAUDE ===


,Ranking,Instituição,UF,Taxa_Execucao
375,1,Prefeitura Municipal de Recife - PE,PE,98.97
353,2,Prefeitura Municipal de Porto Velho - RO,RO,98.64
46,3,Prefeitura Municipal de Belém - PA,PA,98.63
254,4,Prefeitura Municipal de Maceió - AL,AL,97.91
189,5,Prefeitura Municipal de Goiânia - GO,GO,97.61
5,6,Prefeitura Municipal de Aracaju - SE,SE,96.79
415,7,Prefeitura Municipal de Salvador - BA,BA,96.72
127,8,Prefeitura Municipal de Curitiba - PR,PR,96.38
166,9,Prefeitura Municipal de Fortaleza - CE,CE,95.63
275,10,Prefeitura Municipal de Manaus - AM,AM,95.16


In [12]:
# Exemplo: Educacao
print('=== RANKING EDUCACAO ===')
ranking_educacao = ranking_por_funcao(df_taxa_funcao, '12 - Educação')
ranking_educacao.head(10)

=== RANKING EDUCACAO ===


,Ranking,Instituição,UF,Taxa_Execucao
168,1,Prefeitura Municipal de Fortaleza - CE,CE,97.03
67,2,Prefeitura Municipal de Boa Vista - RR,RR,96.98
417,3,Prefeitura Municipal de Salvador - BA,BA,96.43
191,4,Prefeitura Municipal de Goiânia - GO,GO,96.17
355,5,Prefeitura Municipal de Porto Velho - RO,RO,95.84
7,6,Prefeitura Municipal de Aracaju - SE,SE,95.31
312,7,Prefeitura Municipal de Palmas - TO,TO,94.41
109,8,Prefeitura Municipal de Cuiabá - MT,MT,94.35
277,9,Prefeitura Municipal de Manaus - AM,AM,94.05
148,10,Prefeitura Municipal de Florianópolis - SC,SC,94.05


### Interpretação

Os rankings mostram quais capitais são mais eficientes em cada função. Valores mais altos indicam melhor execução orçamentária.

## 3. Estatística Descritiva por Função

Vamos calcular média, mediana, desvio padrão e CV para cada função.

In [13]:
# Funcao para calcular estatisticas por funcao
def estatisticas_por_funcao(df):
    """Calcula estatisticas descritivas para cada funcao."""
    resultados = []
    
    for funcao in df['Funcao'].unique():
        df_funcao = df[df['Funcao'] == funcao].copy()
        df_funcao = df_funcao.dropna(subset=['Taxa_Execucao'])
        
        if len(df_funcao) < 2:
            continue
        
        n = len(df_funcao)
        media = df_funcao['Taxa_Execucao'].mean()
        mediana = df_funcao['Taxa_Execucao'].median()
        dp = df_funcao['Taxa_Execucao'].std()
        cv = (dp / media) * 100 if media > 0 else 0
        
        resultados.append({
            'Funcao': funcao,
            'N': n,
            'Media': round(media, 2),
            'Mediana': round(mediana, 2),
            'Desvio_Padrao': round(dp, 2),
            'CV': round(cv, 2)
        })
    
    return pd.DataFrame(resultados)

df_estatisticas = estatisticas_por_funcao(df_taxa_funcao)
df_estatisticas = df_estatisticas.sort_values('CV', ascending=False)
print('=== ESTATISTICAS POR FUNCAO (ordenadas por CV) ===')
df_estatisticas

=== ESTATISTICAS POR FUNCAO (ordenadas por CV) ===


,Funcao,N,Media,Mediana,Desvio_Padrao,CV
14,19 - Ciência e Tecnologia,20,79.67,89.24,23.68,29.72
6,11 - Trabalho,24,85.50,92.53,22.90,26.78
26,07 - Relações Exteriores,4,85.44,93.84,19.03,22.27
20,20 - Agricultura,16,81.31,81.06,16.73,20.58
24,22 - Indústria,5,84.86,86.88,14.77,17.41
12,17 - Saneamento,24,84.82,86.94,14.44,17.02
18,27 - Desporto e Lazer,26,86.00,89.83,13.02,15.14
11,16 - Habitação,26,84.99,87.18,12.27,14.44
15,23 - Comércio e Serviços,26,85.15,87.26,10.87,12.77
25,05 - Defesa Nacional,3,87.93,84.32,10.73,12.21


### Interpretação

- **CV < 15%**: Baixa dispersão (homogênea)
- **15% <= CV < 30%**: Dispersão moderada
- **CV >= 30%**: Alta dispersão (heterogênea)

Funções com CV alto indicam que as capitais se comportam de forma muito diferente entre si.

### ⚠️ Alerta Estatístico: Funções com Poucas Observações

Ao analisar os dados, encontramos funções com tamanho amostral muito reduzido:

| Função | N (Capitais Declarantes) | Risco |
|--------|--------------------------|-------|
| 05 - Defesa Nacional | 3 | **Muito alto** |
| 07 - Relações Exteriores | 4 | **Alto** |
| 22 - Indústria | 5 | Moderado |
| 25 - Energia | 5 | Moderado |

**Por que isso importa?**
- Com $N < 5$, o **desvio padrão amostral** e o **CV** perdem confiabilidade estatística — qualquer capital extrema distorce completamente a métrica.
- Para essas funções, devemos usar exclusivamente a **mediana** como referência, ignorar rankings e usar os resultados apenas como indicação qualitativa.
- Municípios não são obrigados a gastar em todas as funções (ex: municípios geralmente não têm Defesa Nacional). Isso é estrutural, não um erro nos dados.

**Decisão Metodológica:** Para funções com $N < 5$, os valores de CV e desvio padrão foram calculados e reportados, mas **não serão usados como base de comparação ou ranking nacional**.

## 4. Comparar Média e Mediana

Vamos verificar se a média confirma a mediana ou se há outliers distorcendo.

In [14]:
# Comparar media e mediana
df_estatisticas['Diferenca_Media_Mediana'] = df_estatisticas['Media'] - df_estatisticas['Mediana']
df_estatisticas['Outlier_Possivel'] = df_estatisticas['Diferenca_Media_Mediana'].abs() > 5

print('=== COMPARACAO MEDIA vs MEDIANA ===')
print('Diferenca > 5pp indica possivel outlier\n')
df_estatisticas[['Funcao', 'Media', 'Mediana', 'Diferenca_Media_Mediana', 'Outlier_Possivel']]

=== COMPARACAO MEDIA vs MEDIANA ===
Diferenca > 5pp indica possivel outlier



,Funcao,Media,Mediana,Diferenca_Media_Mediana,Outlier_Possivel
14,19 - Ciência e Tecnologia,79.67,89.24,-9.57,True
6,11 - Trabalho,85.50,92.53,-7.03,True
26,07 - Relações Exteriores,85.44,93.84,-8.40,True
20,20 - Agricultura,81.31,81.06,0.25,False
24,22 - Indústria,84.86,86.88,-2.02,False
12,17 - Saneamento,84.82,86.94,-2.12,False
18,27 - Desporto e Lazer,86.00,89.83,-3.83,False
11,16 - Habitação,84.99,87.18,-2.19,False
15,23 - Comércio e Serviços,85.15,87.26,-2.11,False
25,05 - Defesa Nacional,87.93,84.32,3.61,False


### Interpretação

Se a média é significativamente maior que a mediana, há capitais com valores muito altos puxando a média para cima (outliers positivos).

## 5. Função Mais Homogênea e Mais Heterogênea

In [ ]:
# Funcao mais homogenea (menor CV)
mais_homogenea = df_estatisticas.sort_values('CV').iloc[0]

# Funcao mais heterogenea (maior CV)
mais_heterogenea = df_estatisticas.sort_values('CV', ascending=False).iloc[0]

print('=== ANALISE DE DISPERSAO ===')

print(f'\nMais HOMOGENEA (menor CV = mais parecidas entre si):')
print(f'  Funcao: {mais_homogenea["Funcao"]}')
print(f'  CV: {mais_homogenea["CV"]:.2f}%')
print(f'  Media: {mais_homogenea["Media"]:.2f}%')

print(f'\nMais HETEROGENEA (maior CV = mais desigualdade entre capitais):')
print(f'  Funcao: {mais_heterogenea["Funcao"]}')
print(f'  CV: {mais_heterogenea["CV"]:.2f}%')
print(f'  Media: {mais_heterogenea["Media"]:.2f}%')

## 6. Salvar Resultados

In [16]:
# Salvar estatisticas
df_estatisticas.to_parquet(pasta_processados / 'etapa2_estatisticas_por_funcao.parquet', index=False)

print('Arquivo salvo:')
print(f'  - {pasta_processados / "etapa2_estatisticas_por_funcao.parquet"}')

Arquivo salvo:
  - c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\data\processed\etapa2_estatisticas_por_funcao.parquet


In [17]:
# Fechar conexao com o DuckDB
con.close()
print('Conexao com DuckDB fechada.')


Conexao com DuckDB fechada.


## Conclusão da Etapa 2

### O que descobrimos:

1. **Rankings por função**: Quem executa melhor em cada área
2. **Estatísticas**: Média, mediana, DP e CV para todas as funções
3. **Dispersão**: Qual função tem mais desigualdade entre capitais
4. **Outliers**: Se a média confirma a mediana

### Próxima etapa:

Na **Etapa 3**, vamos **identificar padrões nacionais** — quais capitais estão consistentemente acima da média e qual função é mais problemática.

In [ ]:
print('✓ Notebook 02 executado com sucesso')
print('✓ Comparação de capitais por função')
print(f'✓ Registros processados: {len(df)}')
print(f'✓ Data/hora: {pd.Timestamp.now()}')